<a href="https://colab.research.google.com/github/anujjakhotiya/AI-DL_2026/blob/main/Lab_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Lab/Exp 6: Regularization - Dropout and L2 Weight Decay
# Intel Image Classification using PyTorch

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import zipfile
import shutil
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# ============================================================
# 2. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ============================================================
# 3. UPLOAD KAGGLE ZIP FILE
# ============================================================

from google.colab import files

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

print("Uploaded:", zip_file)


# ============================================================
# 4. EXTRACT DATASET
# ============================================================

extract_path = "/content/intel_dataset"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")


# ============================================================
# 5. FIND TRAINING FOLDER
# ============================================================

train_dir = None

for root, dirs, files_list in os.walk(extract_path):

    # Intel dataset training folder contains the class folders
    class_names = {
        "buildings",
        "forest",
        "glacier",
        "mountain",
        "sea",
        "street"
    }

    if class_names.issubset(set(dirs)):
        train_dir = root
        break

if train_dir is None:
    raise FileNotFoundError(
        "Training folder not found. Check the extracted dataset structure."
    )

print("Training folder:", train_dir)


# ============================================================
# 6. PREPROCESSING
# ============================================================

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(
    train_dir,
    transform=transform
)

print("Classes:", dataset.classes)
print("Total images:", len(dataset))


# ============================================================
# 7. TRAIN-VALIDATION SPLIT
# ============================================================

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))


# ============================================================
# 8. CNN MODEL
# ============================================================

class CNN(nn.Module):

    def __init__(self, dropout=False):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),

            nn.Dropout(0.5) if dropout else nn.Identity(),

            nn.Linear(256, 6)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


# ============================================================
# 9. TRAINING FUNCTION
# ============================================================

def train_model(model, optimizer, name, epochs=5):

    criterion = nn.CrossEntropyLoss()

    train_losses = []
    val_losses = []

    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):

        # -----------------------------
        # Training
        # -----------------------------

        model.train()

        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct / total

        # -----------------------------
        # Validation
        # -----------------------------

        model.eval()

        running_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                running_loss += loss.item()

                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)

                correct += (predicted == labels).sum().item()

        val_loss = running_loss / len(val_loader)
        val_accuracy = 100 * correct / total

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)

        print(
            f"{name} | Epoch [{epoch+1}/{epochs}] "
            f"| Train Loss: {train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f} "
            f"| Train Acc: {train_accuracy:.2f}% "
            f"| Val Acc: {val_accuracy:.2f}%"
        )

    return (
        train_losses,
        val_losses,
        train_accuracies,
        val_accuracies
    )


# ============================================================
# 10. MODEL 1 - DROPOUT
# ============================================================

print("\n==============================")
print("Training Model with Dropout")
print("==============================")

dropout_model = CNN(dropout=True).to(device)

dropout_optimizer = optim.Adam(
    dropout_model.parameters(),
    lr=0.001
)

dropout_results = train_model(
    dropout_model,
    dropout_optimizer,
    "Dropout",
    epochs=5
)


# ============================================================
# 11. MODEL 2 - L2 WEIGHT DECAY
# ============================================================

print("\n==============================")
print("Training Model with L2 Weight Decay")
print("==============================")

l2_model = CNN(dropout=False).to(device)

l2_optimizer = optim.Adam(
    l2_model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

l2_results = train_model(
    l2_model,
    l2_optimizer,
    "L2 Weight Decay",
    epochs=5
)


# ============================================================
# 12. FINAL EVALUATION
# ============================================================

def evaluate_model(model):

    model.eval()

    actual = []
    predicted = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            actual.extend(labels.numpy())
            predicted.extend(preds.cpu().numpy())

    accuracy = accuracy_score(actual, predicted)

    return accuracy, actual, predicted


dropout_accuracy, y_true, dropout_predictions = evaluate_model(
    dropout_model
)

l2_accuracy, _, l2_predictions = evaluate_model(
    l2_model
)


print("\n==============================")
print("FINAL RESULTS")
print("==============================")

print(f"Dropout Accuracy     : {dropout_accuracy * 100:.2f}%")
print(f"L2 Weight Decay      : {l2_accuracy * 100:.2f}%")


# ============================================================
# 13. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report - Dropout")

print(
    classification_report(
        y_true,
        dropout_predictions,
        target_names=dataset.classes
    )
)


print("\nClassification Report - L2 Weight Decay")

print(
    classification_report(
        y_true,
        l2_predictions,
        target_names=dataset.classes
    )
)


# ============================================================
# 14. CONFUSION MATRIX - DROPOUT
# ============================================================

cm_dropout = confusion_matrix(
    y_true,
    dropout_predictions
)

plt.figure(figsize=(7, 6))

plt.imshow(cm_dropout)

plt.title("Confusion Matrix - Dropout")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.colorbar()

plt.xticks(
    range(6),
    dataset.classes,
    rotation=45
)

plt.yticks(
    range(6),
    dataset.classes
)

for i in range(6):
    for j in range(6):

        plt.text(
            j,
            i,
            cm_dropout[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()


# ============================================================
# 15. CONFUSION MATRIX - L2
# ============================================================

cm_l2 = confusion_matrix(
    y_true,
    l2_predictions
)

plt.figure(figsize=(7, 6))

plt.imshow(cm_l2)

plt.title("Confusion Matrix - L2 Weight Decay")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.colorbar()

plt.xticks(
    range(6),
    dataset.classes,
    rotation=45
)

plt.yticks(
    range(6),
    dataset.classes
)

for i in range(6):
    for j in range(6):

        plt.text(
            j,
            i,
            cm_l2[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()


# ============================================================
# 16. LOSS COMPARISON
# ============================================================

dropout_train_loss = dropout_results[0]
dropout_val_loss = dropout_results[1]

l2_train_loss = l2_results[0]
l2_val_loss = l2_results[1]

plt.figure(figsize=(8, 5))

plt.plot(
    dropout_train_loss,
    marker="o",
    label="Dropout - Train"
)

plt.plot(
    dropout_val_loss,
    marker="o",
    label="Dropout - Validation"
)

plt.plot(
    l2_train_loss,
    marker="o",
    label="L2 - Train"
)

plt.plot(
    l2_val_loss,
    marker="o",
    label="L2 - Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Comparison: Dropout vs L2 Weight Decay")

plt.legend()
plt.grid()

plt.show()


# ============================================================
# 17. ACCURACY COMPARISON
# ============================================================

dropout_train_acc = dropout_results[2]
dropout_val_acc = dropout_results[3]

l2_train_acc = l2_results[2]
l2_val_acc = l2_results[3]

plt.figure(figsize=(8, 5))

plt.plot(
    dropout_val_acc,
    marker="o",
    label="Dropout"
)

plt.plot(
    l2_val_acc,
    marker="o",
    label="L2 Weight Decay"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")

plt.title("Validation Accuracy Comparison")

plt.legend()
plt.grid()

plt.show()


# ============================================================
# 18. DISPLAY SAMPLE PREDICTIONS
# ============================================================

images, labels = next(iter(val_loader))

images_gpu = images.to(device)

with torch.no_grad():

    outputs = dropout_model(images_gpu)

    predictions = torch.argmax(outputs, dim=1)

plt.figure(figsize=(12, 6))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    image = images[i].permute(1, 2, 0).numpy()

    # Undo normalization approximately
    image = image * np.array([0.229, 0.224, 0.225])
    image = image + np.array([0.485, 0.456, 0.406])

    image = np.clip(image, 0, 1)

    plt.imshow(image)

    plt.title(
        f"Actual: {dataset.classes[labels[i]]}\n"
        f"Pred: {dataset.classes[predictions[i].cpu()]}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()


# ============================================================
# 19. FINAL COMPARISON
# ============================================================

print("\n==============================")
print("CONCLUSION")
print("==============================")

if dropout_accuracy > l2_accuracy:

    print(
        "Dropout achieved better validation accuracy than "
        "L2 Weight Decay on this dataset."
    )

elif l2_accuracy > dropout_accuracy:

    print(
        "L2 Weight Decay achieved better validation accuracy than "
        "Dropout on this dataset."
    )

else:

    print(
        "Both Dropout and L2 Weight Decay achieved similar "
        "validation accuracy on this dataset."
    )